# WEAPD baselines v2: near-duplicate-free split (MeteCNN + 6 ImageNet models)
Agricultural weather-hazard paper. Same models and training recipes as v1, but the data are cleaned more strictly
before the fixed 8:1:1 split:
* exact duplicates (same MD5) **and near-duplicates** (64-bit DCT perceptual hash, Hamming distance <= 8) are grouped;
* groups whose members carry **different labels** are removed (label conflicts);
* one image per group is kept (the largest), so no image or re-encoded copy of it can appear in two splits.

**Before running:** GPU T4 · Internet On · Add Input → Weather Image Recognition (jehanbhathena).
Results go to `/kaggle/working/results_v2/`.

**Seeds.** `SEEDS=[2, 3, 4]` runs the three remaining seeds in one commit (about 8.5 h on a T4; Kaggle's limit is 12 h).
A time guard stops starting new models after `MAX_HOURS`, so the commit always finishes and saves its output.
The split does not depend on the seed, so results from other notebooks/commits (seeds 0 and 1) are merged afterwards.

In [ ]:
# 0. CONFIG — the only cell you normally edit
CFG = dict(
    MODELS=["metecnn", "resnet50", "efficientnet_b0", "mobilenetv3_large_100",
            "convnext_tiny", "swin_tiny_patch4_window7_224", "vit_small_patch16_224"],
    SEEDS=[2, 3, 4],          # seeds run one after another in this commit (seeds 0 and 1 are already done)
    MAX_HOURS=11.3,           # Kaggle stops a commit at 12 h: no new model is started after this budget
    PHASH_MAX_DIST=8,         # near-duplicate threshold (Hamming distance of 64-bit pHash); verified visually
    SPLIT_SEED=42,            # fixed split shared by every model and seed
    IMG_RESIZE=256,           # Xiao et al. 2021: resize 256, random crop 224
    IMG_SIZE=224,
    BATCH=32,
    EPOCHS=30,                # ImageNet-pretrained models
    LR=1e-4, WD=0.05,         # AdamW + cosine for pretrained models
    PATIENCE=8,               # early stopping on validation macro-F1
    METECNN_EPOCHS=110,       # paper recipe: SGD 0.001, momentum 0.9, wd 5e-4, bs 16, plateau x0.5 / 5 ep
    METECNN_BATCH=16,
    METECNN_PRETRAINED_VGG=False,  # paper trains from scratch; True = init convs from VGG16-BN (ablation)
    SAVE_WEIGHTS_SEEDS=[0],   # keep best checkpoint only for these seeds (saves disk)
    NUM_WORKERS=4,
    DATA_ROOT=None,           # None = auto-detect under /kaggle/input
    WORK="/kaggle/working",
)
print(CFG)

In [ ]:
# 1. Imports and environment
import os, glob, json, time, random, hashlib, shutil, subprocess, sys
import numpy as np, pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True); import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix, classification_report)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T_START = time.time()
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
WORK = CFG["WORK"]; RES = os.path.join(WORK, "results_v2"); os.makedirs(RES, exist_ok=True)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Resume across commits: if an earlier version's output was added as an Input, copy its results here
# (same split.csv, finished runs are then skipped).
prev_runs = []
for prev in glob.glob("/kaggle/input/**/results_v2/runs.csv", recursive=True):
    for f in glob.glob(os.path.join(os.path.dirname(prev), "*")):
        dst = os.path.join(RES, os.path.basename(f))
        if os.path.basename(f) != "runs.csv" and not os.path.exists(dst): shutil.copy(f, dst)
    prev_runs.append(pd.read_csv(prev)); print("Resumed results from", os.path.dirname(prev))
if prev_runs:
    pd.concat(prev_runs).drop_duplicates(["model", "seed"], keep="last").to_csv(os.path.join(RES, "runs.csv"), index=False)

## 1. Find WEAPD
Two ways to give the notebook the data (Add Input on the right):
* **A. Original (recommended for the paper):** upload `dataset(WEAPD).rar` from Harvard Dataverse as a private
  Kaggle dataset. Kaggle keeps .rar files as they are, so this cell extracts it.
* **B. Kaggle copy:** search "Weather Image Recognition" by jehanbhathena (6,862 images). Quicker, but say so in the paper.

In [ ]:
def find_class_root(base):
    """Return the folder whose sub-folders are the 11 classes (most image-holding sub-folders)."""
    best, best_n = None, 0
    for d, subdirs, _ in os.walk(base):
        n = sum(1 for s in subdirs
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(d, s))[:50]))
        if n > best_n:
            best, best_n = d, n
    return best if best_n >= 5 else None

def extract_archives(src, dst):
    os.makedirs(dst, exist_ok=True)
    for a in glob.glob(os.path.join(src, "**", "*.rar"), recursive=True) + \
             glob.glob(os.path.join(src, "**", "*.zip"), recursive=True):
        print("Extracting", a)
        if a.endswith(".zip"):
            shutil.unpack_archive(a, dst)
        else:
            ok = subprocess.run(["7z", "x", "-y", f"-o{dst}", a], capture_output=True).returncode == 0
            if not ok:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rarfile", "unrar"], check=False)
                subprocess.run(["bash", "-c", "apt-get -qq install -y unrar >/dev/null 2>&1 || true"])
                subprocess.run(["unrar", "x", "-o+", a, dst + "/"], check=True)

DATA_ROOT = CFG["DATA_ROOT"] or find_class_root("/kaggle/input")
if DATA_ROOT is None:
    extract_archives("/kaggle/input", os.path.join(WORK, "raw"))
    DATA_ROOT = find_class_root(os.path.join(WORK, "raw"))
assert DATA_ROOT, "WEAPD not found: add the dataset as an Input (Section 1)."
CLASSES = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
print("DATA_ROOT:", DATA_ROOT); print(len(CLASSES), "classes:", CLASSES)

## 2. Clean, remove exact and near-duplicates, resize once, and fix the split
Every readable image gets an MD5 and a 64-bit perceptual hash (DCT of a 32×32 grey image, top-left 8×8, median
threshold). Images with the same MD5 or a pHash Hamming distance <= `PHASH_MAX_DIST` are joined into one group
(union–find). Groups with more than one label are dropped; from every other group the largest image is kept.
Kept images are resized to 256×256 once and cached.

In [ ]:
CACHE = os.path.join(WORK, f"cache{CFG['IMG_RESIZE']}")
split_csv = os.path.join(RES, "split.csv")
_N = 32
_C = np.array([[np.cos(np.pi * (2 * x + 1) * u / (2 * _N)) for x in range(_N)] for u in range(_N)])
_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def phash64(im):
    a = np.asarray(im.convert("L").resize((_N, _N), Image.LANCZOS), dtype=np.float64)
    low = (_C @ a @ _C.T)[:8, :8].flatten()
    return format(int("".join("1" if v else "0" for v in low > np.median(low[1:])), 2), "016x")

def hamming(code, codes):
    return _POP[(codes ^ np.uint64(code)).view(np.uint8).reshape(-1, 8)].sum(1)

if not os.path.exists(split_csv):
    rows, n_bad = [], 0
    for c in CLASSES:
        for f in sorted(os.listdir(os.path.join(DATA_ROOT, c))):
            if not f.lower().endswith(IMG_EXT): continue
            p = os.path.join(DATA_ROOT, c, f)
            try:
                raw = open(p, "rb").read(); im = Image.open(p); im.load(); im = im.convert("RGB")
            except Exception:
                n_bad += 1; continue
            rows.append(dict(src=p, label=c, file=f, md5=hashlib.md5(raw).hexdigest(), phash=phash64(im),
                             w=im.size[0], h=im.size[1]))
    al = pd.DataFrame(rows); n = len(al)
    codes = np.array([int(x, 16) for x in al.phash], dtype=np.uint64)
    par = list(range(n))
    def find(i):
        while par[i] != i: par[i] = par[par[i]]; i = par[i]
        return i
    first = {}
    for i, m in enumerate(al.md5):                       # exact duplicates
        if m in first: par[find(i)] = find(first[m])
        else: first[m] = i
    TH = CFG["PHASH_MAX_DIST"]
    for i in range(n - 1):                               # near-duplicates
        for j in np.nonzero(hamming(codes[i], codes[i + 1:]) <= TH)[0]:
            par[find(i)] = find(i + 1 + int(j))
    al["group"] = [find(i) for i in range(n)]
    gsize = al.group.map(al.group.value_counts()); glabels = al.group.map(al.groupby("group").label.nunique())
    al["exact_dup"] = al.duplicated("md5")
    al["conflict"] = glabels > 1
    al["area"] = al.w * al.h
    keep_idx = al[~al.conflict].sort_values(["group", "area"], ascending=[True, False]).drop_duplicates("group").index
    al["kept"] = al.index.isin(keep_idx)
    al["reason"] = np.where(al.kept, "kept", np.where(al.conflict, "label_conflict",
                            np.where(al.exact_dup, "exact_duplicate", "near_duplicate")))
    audit = al.pivot_table(index="label", columns="reason", values="file", aggfunc="count", fill_value=0)
    audit["original"] = audit.sum(axis=1); audit.loc["TOTAL"] = audit.sum()
    audit.to_csv(os.path.join(RES, "dedup_audit.csv")); al.drop(columns=["area"]).to_csv(os.path.join(RES, "all_images_hashes.csv"), index=False)
    print(f"{n} readable images ({n_bad} unreadable) -> {al.kept.sum()} kept | "
          f"{(al.group.value_counts() > 1).sum()} duplicate groups | {al.conflict.sum()} images in label-conflict groups")
    df = al[al.kept].copy()
    for c in CLASSES: os.makedirs(os.path.join(CACHE, c), exist_ok=True)
    df["path"] = [os.path.join(CACHE, r.label, os.path.splitext(r.file)[0] + ".jpg") for r in df.itertuples()]
    for r in df.itertuples():
        Image.open(r.src).convert("RGB").resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC).save(r.path, quality=95)
    df = df[["path", "label", "src", "md5", "phash", "group"]].reset_index(drop=True)
    tr, tmp = train_test_split(df, test_size=0.2, stratify=df.label, random_state=CFG["SPLIT_SEED"])
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp.label, random_state=CFG["SPLIT_SEED"])
    df["split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"
    df.to_csv(split_csv, index=False)
    print(audit)
df = pd.read_csv(split_csv, dtype={"phash": str})
if not os.path.exists(df.path.iloc[0]):   # cache lost after a restart: rebuild from src
    for r in df.itertuples():
        os.makedirs(os.path.dirname(r.path), exist_ok=True)
        Image.open(r.src).convert("RGB").resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC).save(r.path, quality=95)
# sanity check: no test/val image within the threshold of any train image
_tr = np.array([int(x, 16) for x in df[df.split == "train"].phash], dtype=np.uint64)
_close = sum(int(hamming(int(x, 16), _tr).min() <= CFG["PHASH_MAX_DIST"]) for x in df[df.split != "train"].phash)
print("val/test images with a near-duplicate in train:", _close)
C2I = {c: i for i, c in enumerate(CLASSES)}
counts = df.pivot_table(index="label", columns="split", values="path", aggfunc="count")[["train", "val", "test"]]
counts.loc["TOTAL"] = counts.sum(); counts.to_csv(os.path.join(RES, "class_counts.csv")); counts

In [ ]:
# 3. Data loaders (augmentation follows Xiao et al. 2021: random crop 224, flip, rotation, random erasing)
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = T.Compose([T.RandomCrop(CFG["IMG_SIZE"]), T.RandomHorizontalFlip(), T.RandomRotation(15),
                      T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
tf_eval = T.Compose([T.CenterCrop(CFG["IMG_SIZE"]), T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, d, tf): self.p, self.y, self.tf = d.path.tolist(), [C2I[l] for l in d.label], tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

def loaders(batch, seed):
    g = torch.Generator().manual_seed(seed)
    kw = dict(num_workers=CFG["NUM_WORKERS"], pin_memory=DEVICE == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0)
    return (DataLoader(ImgDS(df[df.split == "train"], tf_train), batch, shuffle=True, drop_last=True, generator=g, **kw),
            DataLoader(ImgDS(df[df.split == "val"], tf_eval), 64, **kw),
            DataLoader(ImgDS(df[df.split == "test"], tf_eval), 64, **kw))

## 4. Models
**MeteCNN** is rebuilt from the paper's description (Xiao et al., 2021, Sec. 3.1): VGG16 layout of 13 3×3 conv layers
(Conv–BN–ReLU), 5 max-pool layers, global average pooling and a softmax classifier with no FC block. The first and last
conv layers use dilation 2; the other 11 carry a squeeze-and-excitation module built from two 1×1 convolutions.
The paper does not give the SE reduction ratio, so we use 16 (Hu et al., 2018) and report it.
Every other model comes from `timm` with ImageNet weights and a new 11-class head.

In [ ]:
class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, max(c // r, 8), 1), nn.ReLU(inplace=True),
                                nn.Conv2d(max(c // r, 8), c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

class MeteCNN(nn.Module):
    CFG16 = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"]
    def __init__(self, n_classes):
        super().__init__()
        layers, cin, idx, n_conv = [], 3, 0, 13
        for v in self.CFG16:
            if v == "M": layers.append(nn.MaxPool2d(2, 2)); continue
            idx += 1
            dil = 2 if idx in (1, n_conv) else 1
            block = [nn.Conv2d(cin, v, 3, padding=dil, dilation=dil, bias=False), nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            if dil == 1: block.append(SE(v))
            layers += block; cin = v
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(512, n_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def forward(self, x): return self.head(torch.flatten(self.pool(self.features(x)), 1))

def load_vgg_init(m):
    """Optional ablation: copy VGG16-BN ImageNet conv/BN weights into MeteCNN (same shapes)."""
    import torchvision
    src = [l for l in torchvision.models.vgg16_bn(weights="IMAGENET1K_V1").features if isinstance(l, (nn.Conv2d, nn.BatchNorm2d))]
    dst = [l for l in m.features if isinstance(l, (nn.Conv2d, nn.BatchNorm2d))]
    for s, d in zip(src, dst): d.load_state_dict(s.state_dict())
    return m

def build(name, pretrained=True):
    if name == "metecnn":
        m = MeteCNN(len(CLASSES))
        return load_vgg_init(m) if CFG["METECNN_PRETRAINED_VGG"] else m
    return timm.create_model(name, pretrained=pretrained, num_classes=len(CLASSES))

def complexity(m):
    params = sum(p.numel() for p in m.parameters()) / 1e6
    try:
        from torch.utils.flop_counter import FlopCounterMode
        m.eval(); x = torch.randn(1, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=next(m.parameters()).device)
        with FlopCounterMode(display=False) as fc: m(x)
        gflops = fc.get_total_flops() / 2e9          # report GMACs-style GFLOPs (multiply-add = 1)
    except Exception:
        gflops = float("nan")
    return params, gflops

In [ ]:
# 5. Train / evaluate one (model, seed) run
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

@torch.no_grad()
def predict(m, dl):
    m.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            P.append(m(x.to(DEVICE, non_blocking=True)).float().softmax(1).cpu())
        Y.append(y)
    P = torch.cat(P).numpy(); return P, torch.cat(Y).numpy()

def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return dict(acc=accuracy_score(y, p), macro_p=pr, macro_r=rc, macro_f1=f1,
                weighted_f1=f1_score(y, p, average="weighted"), kappa=cohen_kappa_score(y, p), mcc=matthews_corrcoef(y, p))

def run(name, seed):
    set_seed(seed)
    is_mete = name == "metecnn"
    bs = CFG["METECNN_BATCH"] if is_mete else CFG["BATCH"]
    epochs = CFG["METECNN_EPOCHS"] if is_mete else CFG["EPOCHS"]
    dl_tr, dl_va, dl_te = loaders(bs, seed)
    m = build(name).to(DEVICE)
    if is_mete:
        opt = torch.optim.SGD(m.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    else:
        opt = torch.optim.AdamW(m.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["LR"], total_steps=epochs * len(dl_tr), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    crit = nn.CrossEntropyLoss()
    best, best_ep, wait, hist, t0 = -1, 0, 0, [], time.time()
    ckpt = os.path.join(RES, f"best_{name}_s{seed}.pt")
    patience = 10**9 if is_mete else CFG["PATIENCE"]   # paper keeps the best-val MeteCNN over all 110 epochs
    for ep in range(1, epochs + 1):
        m.train(); tl, n = 0.0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if not is_mete: sched.step()
            tl += loss.item() * len(y); n += len(y)
        pv, yv = predict(m, dl_va)
        vloss = F.nll_loss(torch.log(torch.tensor(pv) + 1e-9), torch.tensor(yv)).item()
        vf1 = f1_score(yv, pv.argmax(1), average="macro")
        if is_mete: sched.step(vloss)
        hist.append(dict(epoch=ep, train_loss=tl / n, val_loss=vloss, val_acc=accuracy_score(yv, pv.argmax(1)), val_macro_f1=vf1))
        if vf1 > best:
            best, best_ep, wait = vf1, ep, 0; torch.save(m.state_dict(), ckpt)
        else:
            wait += 1
        print(f"  {name} s{seed} ep{ep:3d} train_loss {tl/n:.4f} val_loss {vloss:.4f} val_F1 {vf1:.4f}{' *' if wait == 0 else ''}")
        if wait >= patience: break
    train_min = (time.time() - t0) / 60
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    pt, yt = predict(m, dl_te)
    yp = pt.argmax(1)
    params, gflops = complexity(m)
    m.eval(); x1 = torch.randn(1, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=DEVICE)
    with torch.no_grad():
        for _ in range(10): m(x1)
        if DEVICE == "cuda": torch.cuda.synchronize()
        t1 = time.time()
        for _ in range(50): m(x1)
        if DEVICE == "cuda": torch.cuda.synchronize()
    ms = (time.time() - t1) / 50 * 1000
    out = dict(model=name, seed=seed, best_epoch=best_ep, epochs_run=len(hist), train_min=round(train_min, 1),
               params_M=round(params, 2), gflops=round(gflops, 2), ms_per_img=round(ms, 2), **metrics(yt, yp))
    tag = f"{name}_s{seed}"
    pd.DataFrame(hist).to_csv(os.path.join(RES, f"history_{tag}.csv"), index=False)
    np.save(os.path.join(RES, f"testprob_{tag}.npy"), pt); np.save(os.path.join(RES, "test_labels.npy"), yt)
    rep = classification_report(yt, yp, target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(rep).T.to_csv(os.path.join(RES, f"perclass_{tag}.csv"))
    pd.DataFrame(confusion_matrix(yt, yp), index=CLASSES, columns=CLASSES).to_csv(os.path.join(RES, f"cm_{tag}.csv"))
    if seed not in CFG["SAVE_WEIGHTS_SEEDS"]: os.remove(ckpt)
    runs_csv = os.path.join(RES, "runs.csv")
    old =pd.read_csv(runs_csv) if os.path.exists(runs_csv) else pd.DataFrame()
    pd.concat([old, pd.DataFrame([out])], ignore_index=True).to_csv(runs_csv, index=False)
    print(f"DONE {tag}: acc {out['acc']:.4f} macro-F1 {out['macro_f1']:.4f} ({train_min:.1f} min)")
    del m, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return out

In [ ]:
# 6. Run everything (skips runs already in results/runs.csv)
runs_csv = os.path.join(RES, "runs.csv")
stopped = False
for seed in CFG["SEEDS"]:
    for name in CFG["MODELS"]:
        done = pd.read_csv(runs_csv) if os.path.exists(runs_csv) else pd.DataFrame(columns=["model", "seed"])
        if ((done.model == name) & (done.seed == seed)).any():
            print("skip", name, seed); continue
        need_h = 1.8 if name == "metecnn" else 0.35          # rough run time on a T4
        used_h = (time.time() - T_START) / 3600
        if used_h + need_h > CFG["MAX_HOURS"]:
            print(f"TIME BUDGET: {used_h:.1f} h used, not starting {name} seed {seed}. Re-run the remaining seeds later.")
            stopped = True; break
        print(f"\n=== {name} | seed {seed} ===")
        run(name, seed)
    if stopped: break
print(f"total time {(time.time() - T_START) / 3600:.2f} h")

In [ ]:
# 7. Summary table: mean ± std over seeds (paste into the paper)
r = pd.read_csv(runs_csv)
cols = ["acc", "macro_p", "macro_r", "macro_f1", "weighted_f1", "kappa", "mcc"]
agg = r.groupby("model").agg({**{c: ["mean", "std"] for c in cols}, "params_M": "first", "gflops": "first",
                              "ms_per_img": "mean", "seed": "count"})
table = pd.DataFrame({c: (agg[(c, "mean")] * 100).round(2).astype(str) + " ± " + (agg[(c, "std")].fillna(0) * 100).round(2).astype(str)
                      for c in cols})
table["params (M)"] = agg[("params_M", "first")]; table["GFLOPs"] = agg[("gflops", "first")]
table["ms/img"] = agg[("ms_per_img", "mean")].round(2); table["seeds"] = agg[("seed", "count")]
table = table.loc[agg[("macro_f1", "mean")].sort_values(ascending=False).index]
table.to_csv(os.path.join(RES, "summary_table.csv")); table

In [ ]:
# 8. Per-class F1 of each model (mean over seeds) — shows which hazards are hard (glaze vs frost vs rime)
pcs = []
for f in glob.glob(os.path.join(RES, "perclass_*_s*.csv")):
    if "f1_mean" in f: continue
    name = os.path.basename(f)[9:].rsplit("_s", 1)[0]
    d = pd.read_csv(f, index_col=0).loc[CLASSES, ["f1-score"]].T; d.index = [name]; pcs.append(d)
perclass = (pd.concat(pcs).groupby(level=0).mean() * 100).round(1)
perclass.to_csv(os.path.join(RES, "perclass_f1_mean.csv")); perclass

In [ ]:
# 9. Confusion matrix of the best model (seed 0) + McNemar test vs every other model (seed 0)
import matplotlib.pyplot as plt
from scipy.stats import chi2
best_model = table.index[0]
S0 = int(r.seed.min())   # first seed available
cm = pd.read_csv(os.path.join(RES, f"cm_{best_model}_s{S0}.csv"), index_col=0)
cmn = cm.div(cm.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(len(CLASSES)), CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{cmn.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn.iloc[i, j] > .5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{best_model} (seed {S0}), row-normalised")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig(os.path.join(RES, f"cm_{best_model}.png"), dpi=300); plt.show()

yt = np.load(os.path.join(RES, "test_labels.npy"))
pb = np.load(os.path.join(RES, f"testprob_{best_model}_s{S0}.npy")).argmax(1) == yt
rows = []
for other in table.index[1:]:
    f = os.path.join(RES, f"testprob_{other}_s{S0}.npy")
    if not os.path.exists(f): continue
    po = np.load(f).argmax(1) == yt
    b, c = int((pb & ~po).sum()), int((~pb & po).sum())
    stat = (abs(b - c) - 1) ** 2 / (b + c) if b + c else 0.0
    rows.append(dict(best=best_model, other=other, best_only_right=b, other_only_right=c, chi2=round(stat, 3), p_value=chi2.sf(stat, 1)))
mc = pd.DataFrame(rows); mc.to_csv(os.path.join(RES, "mcnemar_seed0.csv"), index=False); mc

## Outputs (`/kaggle/working/results_v2/`)
| File | Use |
| --- | --- |
| `summary_table.csv` | Main baseline table (mean ± std, params, GFLOPs, ms/img) |
| `perclass_f1_mean.csv` | Per-hazard F1 of every model |
| `cm_<best>.png` | 300-dpi confusion matrix for the paper |
| `mcnemar_seed0.csv` | Significance of the best model vs each baseline |
| `split.csv`, `class_counts.csv` | Exact split used, for reproducibility |
| `dedup_audit.csv`, `all_images_hashes.csv` | Exact/near-duplicate and label-conflict audit per class |
| `runs.csv`, `history_*.csv`, `perclass_*.csv`, `cm_*.csv`, `testprob_*.npy` | Raw per-run results |
| `best_<model>_s0.pt` | Best weights (seed 0) for Grad-CAM and transfer to crop images later |

In [ ]:
# 10. Small zip of results (no weights) for download from the Output tab
import zipfile
zp = os.path.join(WORK, "results_v2_seeds" + "".join(str(x) for x in CFG["SEEDS"]) + "_small.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(RES):
        if not f.endswith(".pt"): z.write(os.path.join(RES, f), f)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 1e6))